# 03 — Multimodal Alignment

Demonstrate CLIP-style contrastive alignment between geometry and text modalities.

**Topics:**
- GeoFusionModel architecture
- Contrastive loss (NT-Xent) in action
- Joint embedding space
- Cross-modal similarity

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from geofusion.models.pointnet2 import PointNet2Encoder
from geofusion.models.text_encoder import SimpleTextEncoder
from geofusion.models.multimodal import MultimodalAligner, GeoFusionModel
from geofusion.training.losses import NTXentLoss

## Build Multimodal Model

In [ ]:
EMBED_DIM = 128

geo_encoder = PointNet2Encoder(embed_dim=EMBED_DIM, use_normals=False)
txt_encoder = SimpleTextEncoder(vocab_size=5000, embed_dim=EMBED_DIM)

model = GeoFusionModel(
    geometry_encoder=geo_encoder,
    text_encoder=txt_encoder,
    embed_dim=EMBED_DIM,
    num_classes=10,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

## Contrastive Loss Demo

In [ ]:
aligner = MultimodalAligner(embed_dim=EMBED_DIM, temperature=0.07)
loss_fn = NTXentLoss(temperature=0.07)

# Simulate aligned pairs
batch_size = 16
geo_emb = torch.randn(batch_size, EMBED_DIM)
txt_emb = torch.randn(batch_size, EMBED_DIM)

# Before alignment — random embeddings yield high loss
loss_before = aligner(geo_emb, txt_emb)
print(f'Loss (random embeddings): {loss_before.item():.4f}')

# Simulate "aligned" embeddings — same with noise
shared = torch.randn(batch_size, EMBED_DIM)
geo_aligned = shared + 0.1 * torch.randn_like(shared)
txt_aligned = shared + 0.1 * torch.randn_like(shared)

loss_aligned = aligner(geo_aligned, txt_aligned)
print(f'Loss (aligned embeddings): {loss_aligned.item():.4f}')

## Cross-Modal Similarity Matrix

In [ ]:
import torch.nn.functional as F

n = 8
# Create mock aligned embeddings
base = torch.randn(n, EMBED_DIM)
geo_embs = F.normalize(base + 0.05 * torch.randn_like(base), dim=1)
txt_embs = F.normalize(base + 0.05 * torch.randn_like(base), dim=1)

sim_matrix = geo_embs @ txt_embs.T

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix.detach().numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Text Embeddings', fontsize=12)
ax.set_ylabel('Geometry Embeddings', fontsize=12)
ax.set_title('Cross-Modal Similarity Matrix', fontsize=14)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print('Diagonal (matched pairs) should be high:')
print(f'  Mean diagonal: {sim_matrix.diag().mean():.4f}')
print(f'  Mean off-diagonal: {(sim_matrix.sum() - sim_matrix.diag().sum()) / (n*n - n):.4f}')